# LandShift D3.5 — Section 4.2
# Ecosystem Health

| | |
|---|---|
| **Project** | LandShift — Horizon Europe, Grant Agreement No. 101182007 |
| **Task** | T3.3 — Mapping ecosystem services linked with past, current, and projected LULUCF |
| **Lead** | University of Geneva (UNIGE) |
| **Collaborators** | POLIMI, UNIBAS, ECoE, BFRS |
| **Reference** | D3.5 Ecosystem services & landscape ecology, §4.2 |
| **FAIR compliance** | Outputs are open CSV / PNG / GeoTIFF / MD; notebook is fully reproducible |

---

## Purpose

This notebook implements the **Tier 3 LandShift product** described in §4.2 of D3.5.  
It integrates two Tier-2 outputs that are assumed to have already been computed
(see the [WP3 folder of the HE-LandShift repository](https://github.com/LivingEarthLab/HE-LandShift/tree/main/WP3)):

1. **MAES ecosystem extent raster** — integer raster classifying every pixel into a MAES Level-2 ecosystem type (`<name>_maesL2.tif`, from `ecosystem_extent/ecosystem_extent.ipynb`).
2. **SEEA-EA ecosystem condition raster** — 1–5 score raster (`ecosystem_condition_map.tif`, from `ecosystem_condition/ecosystem_condition.ipynb`).

> **Methodology update — ESAI is no longer used.**  Ecosystem condition now follows the SEEA-EA workflow of the
> WP3 repository: a weighted composite degradation score built from four biophysical indicators
> (NDVI deviation from the 2000–2015 reference period, canopy-cover deficit, fragmentation, human pressure),
> classified on the 5-point SEEA-EA scale (**5 = Very Good … 1 = Very Poor**).
> Higher scores therefore mean *better* condition — the opposite orientation to ESAI.
> Only natural ecosystems are scored upstream; cropland, urban and water are NoData in the condition raster
> and are reported here as **not assessed**.

The notebook then:
- Validates the condition raster (class codes 1–5) and, if needed, re-grids it to the MAES grid with **nearest-neighbour** resampling (the layer is categorical)
- Builds the **SEEA-EA Ecosystem Condition Account** (Tables 4–6 of D3.5)
- Derives three complementary **Ecosystem Health Indicators** per ecosystem type (§4.2.1.4):
  - **A.** Mean Condition Score (1–5, area-weighted mean per ecosystem type)
  - **B.** Percentage of assessed area in Poor / Very Poor condition (classes 1–2)
  - **C.** Normalised Ecosystem Health Index (0–1 scale)
- Exports the **Ecosystem Health Index as GeoTIFFs** (pixel level and ecosystem-type level)
- Produces all supporting **spatial maps and charts**
- Supports **multi-year / multi-period** inputs for trend analysis (§4.2.1.5)
- Runs a full set of **quality checks** (§4.2.3)
- Exports all tables, figures, and a Markdown processing report

### Five-step workflow (D3.5 §4.2.1)

| Step | Description | D3.5 Reference |
|------|-------------|----------------|
| 1 | Establish the Ecosystem Extent Layer (MAES) | §4.2.1.1 |
| 2 | Load and validate the SEEA-EA Condition Layer | §4.2.1.2 |
| 3 | Spatial Overlay and Attribution | §4.2.1.3 |
| 4 | Derive Ecosystem Health Indicators | §4.2.1.4 |
| 5 | Integration into SEEA-EA Condition Accounts | §4.2.1.5 |

---

## How to use this notebook

1. Run **Section 0** to install and import dependencies.
2. Edit **Section 1** (Configuration) — set `MAES_RASTER`, `CONDITION_PERIODS`, etc.
3. Run all cells top-to-bottom. Outputs land in `OUTPUT_DIR`.
4. To test without real data, keep `DEMO_MODE = True` (Section A generates synthetic rasters).

> ℹ️  **Relationship to §4.1 and WP3:** This notebook re-uses the MAES extent layer produced in §4.1 (Step 1)
> and the condition layer produced by `ecosystem_condition.ipynb`.  If you have already run those notebooks,
> simply point `MAES_RASTER` and `CONDITION_PERIODS` to their output files.

---
## Section 0 — Dependencies

In [23]:
# Install required packages if not already present.
# Comment out this cell if working in a pre-configured environment.
import subprocess, sys

PACKAGES = ["numpy", "pandas", "rasterio", "matplotlib", "scipy", "affine"]
for pkg in PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

print("All packages ready.")

All packages ready.


In [ ]:
import datetime
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter, PercentFormatter

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# display() is a Jupyter built-in; provide a plain-print fallback so cells
# also work when the notebook is executed as a plain Python script.
try:
    from IPython.display import display
except ImportError:
    def display(obj):  # noqa: F811
        print(obj)

print(f"numpy    {np.__version__}")
print(f"pandas   {pd.__version__}")
print(f"rasterio {rasterio.__version__}")
print("Imports OK.")

---
## Section 1 — Configuration

**Edit the variables in this cell before running the rest of the notebook.**

In [ ]:
# ── Input — MAES ecosystem extent raster (Tier-2 output, same as used in §4.1)
#   dtype  : integer (uint8 / int16)
#   values : MAES Level-2 class codes
#   CRS    : any projected CRS with metre units (e.g. EPSG:3035)
MAES_RASTER = Path("PATH_TO_MAES_RASTER")

# Lookup table — required columns: 'maes_code' (int), 'ecosystem_type' (str)
#                optional column : 'maes_color' (hex string, e.g. '#2ca25f')
MAES_LOOKUP_CSV = Path("PATH_TO/maes_lookup.csv")

# ── Input — SEEA-EA ecosystem condition rasters
#   Output of WP3 `ecosystem_condition/ecosystem_condition.ipynb`
#   (`ecosystem_condition_map.tif`).
#
#   values : 5 = Very Good, 4 = Good, 3 = Fair, 2 = Poor, 1 = Very Poor / Degraded
#   no-data: -9999  (natural ecosystems only are scored upstream; cropland, urban,
#            water and ice are NoData and therefore "not assessed" in this notebook)
#
# Supply one entry per accounting period to enable trend analysis (§4.2.1.5).
# Keys   : integer year labels used in table columns, filenames, and figure titles.
# For a single-period assessment, provide just one entry.
CONDITION_PERIODS = {
    2017: Path("PATH_TO_ECOSYSTEM_CONDITION_YEAR1"),
    2025: Path("PATH_TO_ECOSYSTEM_CONDITION_YEAR2"),
}

# ── Output
OUTPUT_DIR      = Path("outputs/") #TBC
ACCOUNTING_YEAR = 2025   # primary (most recent) year for stand-alone tables/maps TBC
LIVING_SPACE    = "Basilicata"   #TBC

# ── Spatial settings
# True  → re-grid condition rasters to the MAES grid when CRS / resolution / extent differ.
# False → assume all rasters are already perfectly co-registered.
ALIGN_CONDITION        = True
# The condition layer is CATEGORICAL (classes 1–5): keep nearest-neighbour.
# Bilinear/cubic resampling would create non-existent intermediate classes.
RESAMPLE_METHOD        = Resampling.nearest
PIXEL_AREA_HA_OVERRIDE = None   # set e.g. 0.01 to force a fixed pixel area
MAES_EXCLUDE_CODES     = [0, 255]
CONDITION_NODATA       = -9999.0

# ── SEEA-EA condition classes (as produced by the WP3 condition workflow)
#    The class boundaries, indicator weights and the 2000–2015 reference period are
#    defined upstream in `ecosystem_condition.ipynb` — do not re-classify here.
COND_LABELS  = {5: "Very Good",
                4: "Good",
                3: "Fair",
                2: "Poor",
                1: "Very Poor / Degraded"}
COND_KEYS    = {5: "very_good", 4: "good", 3: "fair", 2: "poor", 1: "very_poor"}
COND_COLOURS = {5: "#1a7a1a",   # same palette as the WP3 condition map
                4: "#78c850",
                3: "#f0d040",
                2: "#e87820",
                1: "#c82020"}
COND_MIN, COND_MAX = 1, 5

# Classes counted as "degraded" for indicator B (Poor + Very Poor).
# Same definition as `pct_poor_degraded` in the WP3 RLE Criterion D assessment.
DEGRADED_CLASSES = (1, 2)

# ── Health index (§4.2.1.4 — indicator C)
#    Health = (mean score − COND_MIN) / (COND_MAX − COND_MIN)  →  1 = best, 0 = worst.
#    The ecosystem-type status label is the SEEA-EA class nearest to the mean score
#    (e.g. mean 3.6 → "Good"), i.e. class boundaries at 1.5 / 2.5 / 3.5 / 4.5.

# ── Trend settings (§4.2.1.5)
TREND_STABLE_TOL = 0.05   # |Δ mean score| below this → "Stable"

# ── Quality-check setting (§4.2.3)
MIN_ASSESSED_PCT = 90.0   # assessed ecosystem types with less assessed area are flagged

# ── Demo mode
# True  → synthetic rasters are generated and used; real paths above are ignored.
# False → production run; all paths must point to real files.
DEMO_MODE = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Living Space      : {LIVING_SPACE}")
print(f"  Accounting year   : {ACCOUNTING_YEAR}")
print(f"  Condition periods : {list(CONDITION_PERIODS.keys())}")
print(f"  Output folder     : {OUTPUT_DIR.resolve()}")
print(f"  Demo mode         : {DEMO_MODE}")

---
## Section A — Synthetic Demo Data *(skip in production)*

When `DEMO_MODE = True` this section generates small synthetic GeoTIFFs that mimic
real MAES and SEEA-EA condition outputs, so the full notebook can be executed and validated
without any input files.  Like the real WP3 condition raster, the synthetic condition maps
score **natural ecosystems only** (forest, grassland, wetlands); cropland and urban are NoData.
All configuration paths are overwritten automatically.
Set `DEMO_MODE = False` in Section 1 to run on real data.

In [ ]:
if DEMO_MODE:
    from rasterio.transform import from_origin
    from scipy.ndimage import gaussian_filter

    DEMO_DIR = OUTPUT_DIR / "_demo_inputs"
    DEMO_DIR.mkdir(parents=True, exist_ok=True)

    ROWS, COLS = 200, 200
    DEMO_RES   = 10.0          # metres
    DEMO_TF    = from_origin(2600000, 1200000, DEMO_RES, DEMO_RES)
    DEMO_CRS   = "EPSG:3035"
    rng        = np.random.default_rng(42)

    # ── MAES raster: 5 ecosystem types in horizontal bands + salt-pepper ──────
    maes_codes_demo = np.array([1, 2, 3, 4, 5], dtype=np.uint8)
    maes_labels_demo = {1: "Forest and woodlands", 2: "Cropland",
                        3: "Grassland",            4: "Wetlands", 5: "Urban"}
    band_h = ROWS // len(maes_codes_demo)
    maes_arr_demo = np.zeros((ROWS, COLS), dtype=np.uint8)
    for i, code in enumerate(maes_codes_demo):
        maes_arr_demo[i * band_h:(i + 1) * band_h, :] = code
    noise = rng.random((ROWS, COLS)) < 0.08
    maes_arr_demo[noise] = rng.choice(maes_codes_demo, size=noise.sum())

    demo_maes_path = DEMO_DIR / "maes_level2_demo.tif"
    with rasterio.open(demo_maes_path, "w",
                       driver="GTiff", height=ROWS, width=COLS, count=1,
                       dtype=np.uint8, crs=DEMO_CRS,
                       transform=DEMO_TF, nodata=0) as dst:
        dst.write(maes_arr_demo, 1)

    # ── MAES lookup CSV ───────────────────────────────────────────────────────
    demo_lookup_path = DEMO_DIR / "maes_lookup_demo.csv"
    pd.DataFrame({
        "maes_code"     : list(maes_labels_demo.keys()),
        "ecosystem_type": list(maes_labels_demo.values()),
        "maes_color"    : ["#2ca25f", "#d8b365", "#fee08b", "#4575b4", "#d73027"],
    }).to_csv(demo_lookup_path, index=False)

    # ── SEEA-EA condition rasters (1–5) for two periods ───────────────────────
    # Natural ecosystems only: forest (1), grassland (3), wetlands (4).
    # Cropland (2) and Urban (5) stay NoData, as in the WP3 condition workflow.
    # Mean score per ecosystem and period (higher = better): forest improves,
    # grassland degrades.  A spatially smooth field makes the maps look realistic.
    cond_means = {
        2018: {1: 3.8, 3: 3.2, 4: 3.5},
        2024: {1: 4.1, 3: 2.9, 4: 3.4},
    }
    base_field = gaussian_filter(rng.normal(size=(ROWS, COLS)), sigma=6)
    base_field = base_field / base_field.std()

    demo_cond_paths = {}
    for yr, means in cond_means.items():
        field    = base_field + 0.35 * rng.normal(size=(ROWS, COLS))
        cond_arr = np.full((ROWS, COLS), CONDITION_NODATA, dtype=np.float32)
        for code, mu in means.items():
            m = (maes_arr_demo == code)
            cond_arr[m] = np.rint(mu + 0.9 * field[m]).clip(1, 5).astype(np.float32)
        # ~2 % random gaps inside the natural ecosystems (e.g. cloud / no-data pixels)
        gaps = (rng.random((ROWS, COLS)) < 0.02) & (cond_arr != CONDITION_NODATA)
        cond_arr[gaps] = CONDITION_NODATA

        path = DEMO_DIR / f"ecosystem_condition_map_{yr}_demo.tif"
        with rasterio.open(path, "w",
                           driver="GTiff", height=ROWS, width=COLS, count=1,
                           dtype=np.float32, crs=DEMO_CRS,
                           transform=DEMO_TF, nodata=CONDITION_NODATA) as dst:
            dst.write(cond_arr, 1)
        demo_cond_paths[yr] = path

    # ── Override config ───────────────────────────────────────────────────────
    MAES_RASTER       = demo_maes_path
    MAES_LOOKUP_CSV   = demo_lookup_path
    CONDITION_PERIODS = demo_cond_paths
    ALIGN_CONDITION   = False   # demo rasters are already co-registered

    print("Demo rasters created:")
    print(f"  MAES      : {MAES_RASTER}")
    for yr, p in CONDITION_PERIODS.items():
        print(f"  Condition {yr}: {p.name}")
else:
    print("DEMO_MODE = False — using paths defined in Section 1.")

---
## Section 2 — Helper Functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2.1  Pixel area
# ─────────────────────────────────────────────────────────────────────────────
def get_pixel_area_ha(raster_path: Path) -> float:
    """
    Return pixel area in hectares derived from the raster affine transform.
    Assumes a projected CRS whose linear unit is the metre (e.g. EPSG:3035).
    Overridden by PIXEL_AREA_HA_OVERRIDE when set.
    """
    if PIXEL_AREA_HA_OVERRIDE is not None:
        return float(PIXEL_AREA_HA_OVERRIDE)
    with rasterio.open(raster_path) as src:
        return (abs(src.transform.a) * abs(src.transform.e)) / 10_000.0


# ─────────────────────────────────────────────────────────────────────────────
# 2.2  Spatial alignment (categorical raster → nearest neighbour)
# ─────────────────────────────────────────────────────────────────────────────
def align_to_maes(
    src_path: Path,
    ref_path: Path,
    dst_path: Path,
    resampling: Resampling = Resampling.nearest,
    nodata: float = CONDITION_NODATA,
) -> Path:
    """
    Reproject and resample src_path to exactly match the CRS, resolution,
    and extent of ref_path (the MAES raster).  Writes result to dst_path.

    The SEEA-EA condition layer is a categorical raster (classes 1–5), so
    nearest-neighbour resampling is the default; bilinear/cubic would generate
    fractional values that do not correspond to any condition class.

    Implements the §4.2.3 requirement:
    'Maintain consistent spatial resolution [between MAES and condition layers].'
    """
    with rasterio.open(ref_path) as ref:
        ref_crs, ref_tf = ref.crs, ref.transform
        ref_h,   ref_w  = ref.height, ref.width

    with rasterio.open(src_path) as src:
        raw    = src.read(1).astype(np.float32)
        src_nd = src.nodata if src.nodata is not None else nodata
        raw    = np.where(np.isclose(raw, src_nd) | np.isnan(raw), nodata, raw)
        dst_data = np.full((ref_h, ref_w), nodata, dtype=np.float32)
        reproject(
            source=raw, destination=dst_data,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref_tf,        dst_crs=ref_crs,
            resampling=resampling,
            src_nodata=nodata,           dst_nodata=nodata,
        )

    with rasterio.open(dst_path, "w",
                       driver="GTiff", height=ref_h, width=ref_w, count=1,
                       dtype=np.float32, crs=ref_crs,
                       transform=ref_tf, nodata=nodata) as dst:
        dst.write(dst_data, 1)
    return dst_path


# ─────────────────────────────────────────────────────────────────────────────
# 2.3  Condition-raster validation
# ─────────────────────────────────────────────────────────────────────────────
def sanitize_condition(arr: np.ndarray):
    """
    Keep only valid SEEA-EA class codes (the keys of COND_LABELS, i.e. 1–5).
    Any other finite value is set to NaN and counted.

    Returns
    -------
    clean     : float32 array, NaN = not assessed
    n_invalid : number of finite pixels that were not a valid class code
    """
    finite    = ~np.isnan(arr)
    ok        = finite & np.isin(arr, list(COND_LABELS))
    n_invalid = int((finite & ~ok).sum())
    clean     = np.where(ok, arr, np.nan).astype(np.float32)
    return clean, n_invalid


# ─────────────────────────────────────────────────────────────────────────────
# 2.4  Zonal condition statistics per MAES ecosystem type  (implements §4.2.1.3)
# ─────────────────────────────────────────────────────────────────────────────
def zonal_condition_stats(
    maes_arr    : np.ndarray,
    cond_arr    : np.ndarray,
    pixel_ha    : float,
    unique_codes: np.ndarray,
    excl_mask   : np.ndarray,
) -> pd.DataFrame:
    """
    For each MAES ecosystem type i, compute (on assessed pixels only):

        Mean score_i = Σ(score_p × A_p) / Σ A_p          [D3.5 §4.2.1.3]

    Pixels have a uniform area in a projected CRS, so the area-weighted mean
    equals the arithmetic mean.  Also returns descriptive statistics, the area
    and share of each SEEA-EA class, and the share of the ecosystem type that
    was actually assessed (cropland / urban / water are not scored upstream).

    Returns a DataFrame with one row per ecosystem type.  Ecosystem types
    without any assessed pixel have NaN statistics ("not assessed").
    """
    records = []
    for code in unique_codes:
        zone    = (maes_arr == code) & (~excl_mask)
        vals    = cond_arr[zone]
        valid   = vals[~np.isnan(vals)]
        n_pix   = int(zone.sum())
        n_valid = int(valid.size)

        rec = {
            "maes_code"       : int(code),
            "pixel_count"     : n_pix,
            "area_ha"         : round(n_pix   * pixel_ha, 4),
            "area_assessed_ha": round(n_valid * pixel_ha, 4),
            "pct_assessed"    : round(n_valid / n_pix * 100, 2) if n_pix else np.nan,
            "mean_score"      : round(float(valid.mean()),       4) if n_valid else np.nan,
            "median_score"    : round(float(np.median(valid)),   4) if n_valid else np.nan,
            "std_score"       : round(float(valid.std()),        4) if n_valid else np.nan,
            "min_score"       : round(float(valid.min()),        4) if n_valid else np.nan,
            "max_score"       : round(float(valid.max()),        4) if n_valid else np.nan,
        }
        for cc, key in COND_KEYS.items():
            n_cc = int(np.sum(valid == cc))
            rec[f"area_ha_{key}"] = round(n_cc * pixel_ha, 4)
            rec[f"pct_{key}"]     = round(n_cc / n_valid * 100, 2) if n_valid else np.nan

        # Indicator B — share of assessed area in Poor + Very Poor condition
        n_deg = int(np.isin(valid, DEGRADED_CLASSES).sum())
        rec["pct_poor_degraded"] = round(n_deg / n_valid * 100, 2) if n_valid else np.nan
        records.append(rec)
    return pd.DataFrame(records)


# ─────────────────────────────────────────────────────────────────────────────
# 2.5  Ecosystem Health Index (indicator C, §4.2.1.4)
# ─────────────────────────────────────────────────────────────────────────────
def compute_health_index(score):
    """
    Normalise a SEEA-EA condition score (scalar, Series or array) to a 0–1
    Health Index where 1 = best condition:

        Health = (score − COND_MIN) / (COND_MAX − COND_MIN)

    The SEEA-EA scale is already oriented "higher = better", so no inversion
    is needed.
    NaN (= not assessed) is propagated.
    """
    return np.round(np.clip((score - COND_MIN) / (COND_MAX - COND_MIN), 0, 1), 4)


# ─────────────────────────────────────────────────────────────────────────────
# 2.6  Health status label
# ─────────────────────────────────────────────────────────────────────────────
def health_label(mean_score: float) -> str:
    """Map a mean condition score to the nearest SEEA-EA class label (D3.5 Table 5)."""
    if pd.isna(mean_score):
        return "Not assessed"
    cls = int(np.clip(np.floor(mean_score + 0.5), COND_MIN, COND_MAX))
    return COND_LABELS[cls]


# ─────────────────────────────────────────────────────────────────────────────
# 2.6b  D3.5 Table 5 three-level status (Good / Moderate / Degraded)
# ─────────────────────────────────────────────────────────────────────────────
def health_label_d35(mean_score: float) -> str:
    """
    Three-level status used in D3.5 Table 5, derived from the mean SEEA-EA score:
        mean >= 3.5  → Good      (classes 4–5)
        2.5 - 3.5    → Moderate  (class 3)
        mean <  2.5  → Degraded  (classes 1–2, consistent with indicator B)
    """
    if pd.isna(mean_score):
        return "Not assessed"
    if mean_score >= 3.5:
        return "Good"
    return "Moderate" if mean_score >= 2.5 else "Degraded"


# ─────────────────────────────────────────────────────────────────────────────
# 2.7  Plotting helpers
# ─────────────────────────────────────────────────────────────────────────────
FIGURE_DPI = 150
NOT_ASSESSED_RGB = (0.88, 0.88, 0.88)

def save_fig(fig, path: Path) -> None:
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"   Figure saved → {path.name}")


def get_cmap_n(name: str, n: int = None):
    """Return a Matplotlib colormap (optionally resampled to n colours)."""
    cmap = matplotlib.colormaps[name]
    return cmap.resampled(n) if n else cmap


def make_maes_rgb(maes_arr, extent_df):
    """Build an (H,W,3) RGB image and legend patches from the MAES extent table."""
    fallback = get_cmap_n("tab20", len(extent_df))
    rgb, patches = np.zeros((*maes_arr.shape, 3)), []
    for i, row in extent_df.iterrows():
        colour = (
            mcolors.to_rgb(row["maes_color"])
            if "maes_color" in extent_df.columns and pd.notna(row.get("maes_color"))
            else fallback(i)[:3]
        )
        rgb[maes_arr == row["maes_code"]] = colour
        patches.append(mpatches.Patch(color=colour, label=row["ecosystem_type"]))
    return rgb, patches


def make_cond_rgb(cond_arr):
    """
    Build an (H,W,3) RGB image and legend patches from a SEEA-EA condition array
    (5 = Very Good … 1 = Very Poor).  NaN (not assessed) is drawn in light grey.
    """
    rgb = np.empty((*cond_arr.shape, 3))
    rgb[:] = NOT_ASSESSED_RGB
    patches = []
    for cc, cl in COND_LABELS.items():
        colour = mcolors.to_rgb(COND_COLOURS[cc])
        rgb[cond_arr == cc] = colour
        patches.append(mpatches.Patch(color=colour, label=f"{cc} – {cl}"))
    patches.append(mpatches.Patch(color=NOT_ASSESSED_RGB, label="Not assessed"))
    return rgb, patches


print("Helper functions defined.")

---
## Step 1 — Ecosystem Extent Layer (MAES)

**D3.5 §4.2.1.1**

The MAES raster is read and summarised into the extent account that defines
the spatial accounting units to which ecosystem condition will be attributed.

In [ ]:
# ── 1a. Load lookup table ─────────────────────────────────────────────────────
maes_lookup = pd.read_csv(MAES_LOOKUP_CSV)
assert {"maes_code", "ecosystem_type"}.issubset(maes_lookup.columns), (
    "MAES lookup CSV must contain at least 'maes_code' and 'ecosystem_type'."
)
code_to_name = dict(zip(maes_lookup["maes_code"], maes_lookup["ecosystem_type"]))
print("MAES lookup:")
display(maes_lookup)

# ── 1b. Read MAES raster ──────────────────────────────────────────────────────
with rasterio.open(MAES_RASTER) as src:
    maes_arr  = src.read(1)      # kept in memory throughout the notebook
    maes_crs  = src.crs
    maes_tf   = src.transform
    nd_maes   = src.nodata

pix_ha   = get_pixel_area_ha(MAES_RASTER)
maes_res = abs(maes_tf.a)
print(f"\nCRS        : {maes_crs}")
print(f"Resolution : {maes_res:.1f} m")
print(f"Pixel area : {pix_ha:.6f} ha")
print(f"Shape      : {maes_arr.shape}")

# ── 1c. Exclusion mask ────────────────────────────────────────────────────────
excl_mask = np.isin(maes_arr, MAES_EXCLUDE_CODES)
if nd_maes is not None:
    excl_mask |= (maes_arr == int(nd_maes))

# ── 1d. Ecosystem Extent Account (SEEA-EA Table 4) ───────────────────────────
unique_codes, counts = np.unique(maes_arr[~excl_mask], return_counts=True)

extent_table = pd.DataFrame({
    "maes_code"  : unique_codes.astype(int),
    "pixel_count": counts,
    "area_ha"    : (counts * pix_ha).round(2),
})
extent_table["ecosystem_type"] = (
    extent_table["maes_code"].map(code_to_name).fillna("Unknown")
)
if "maes_color" in maes_lookup.columns:
    extent_table["maes_color"] = extent_table["maes_code"].map(
        dict(zip(maes_lookup["maes_code"], maes_lookup["maes_color"]))
    )
total_area = extent_table["area_ha"].sum()
extent_table["area_pct"] = (extent_table["area_ha"] / total_area * 100).round(2)

leading = ["maes_code", "ecosystem_type", "pixel_count", "area_ha", "area_pct"]
extra   = [c for c in extent_table.columns if c not in leading]
extent_table = extent_table[leading + extra]

print(f"\nTotal mapped area : {total_area:,.2f} ha")
print(f"Ecosystem types   : {len(extent_table)}")
print("\n📋 Table 4 — SEEA-EA Ecosystem Extent Account (for condition attribution):")
display(extent_table)

out_t4 = OUTPUT_DIR / f"table4_ecosystem_extent_{ACCOUNTING_YEAR}.csv"
extent_table.to_csv(out_t4, index=False)
print(f"Saved → {out_t4.name}")

In [ ]:
# ── 1e. MAES extent map ───────────────────────────────────────────────────────
rgb_maes, patches_maes = make_maes_rgb(maes_arr, extent_table)

fig, (ax_map, ax_bar) = plt.subplots(
    1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [2, 1]}
)
ax_map.imshow(rgb_maes, interpolation="nearest")
ax_map.set_title(
    f"MAES Ecosystem Extent — {LIVING_SPACE} ({ACCOUNTING_YEAR})",
    fontsize=12, fontweight="bold"
)
ax_map.legend(handles=patches_maes, loc="lower right", fontsize=8,
              title="Ecosystem Type", framealpha=0.85)
ax_map.axis("off")

bar_colours_maes = [
    mcolors.to_rgb(row["maes_color"])
    if "maes_color" in extent_table.columns and pd.notna(row.get("maes_color"))
    else get_cmap_n("tab20")(i / max(len(extent_table) - 1, 1))[:3]
    for i, (_, row) in enumerate(extent_table.iterrows())
]
bars = ax_bar.barh(
    extent_table["ecosystem_type"], extent_table["area_ha"],
    color=bar_colours_maes, edgecolor="grey", linewidth=0.5
)
ax_bar.set_xlabel("Area (ha)", fontsize=10)
ax_bar.set_title("Area by Ecosystem Type", fontsize=11, fontweight="bold")
ax_bar.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax_bar.xaxis.get_major_formatter().set_powerlimits((0, 0))
for bar, pct in zip(bars, extent_table["area_pct"]):
    ax_bar.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
                f"{pct:.1f}%", va="center", fontsize=8)
ax_bar.margins(x=0.18)
plt.tight_layout()
save_fig(fig, OUTPUT_DIR / f"fig1_maes_extent_{ACCOUNTING_YEAR}.png")
plt.show()

---
## Step 2 — Load and Validate the SEEA-EA Condition Layers

**D3.5 §4.2.1.2**

The condition rasters are produced upstream by the WP3 workflow
(`ecosystem_condition_prep.ipynb` → `clip_rasters_with_vector.ipynb` → `ecosystem_condition.ipynb`).
Each raster is:
1. Checked and (if needed) re-gridded to the MAES grid — **nearest neighbour**, because the layer is categorical.
2. Validated: only class codes 1–5 are kept; other values are set to *not assessed*.

The upstream composite degradation score (reference period **2000–2015**) is

$$D = 0.40\,\text{NDVI}_{dev} + 0.25\,\text{Canopy}_{def} + 0.20\,\text{Fragmentation} + 0.15\,\text{Pressure}$$

and is classified on the 5-point SEEA-EA scale (default weights and thresholds of the WP3 notebook):

| Score | Class | Composite degradation $D$ |
|---|---|---|
| **5** | Very Good | ≤ 0.05 |
| **4** | Good | 0.05 – 0.25 |
| **3** | Fair | 0.25 – 0.50 |
| **2** | Poor | 0.50 – 0.75 |
| **1** | Very Poor / Degraded | > 0.75 |

Only **natural ecosystems** are scored; cropland, urban, water and ice are NoData and are reported as *not assessed*.

*Source: HE-LandShift repository, `WP3/ecosystem_condition/ecosystem_condition.ipynb`; UN SEEA EA (2021); see D3.5 §3.2*

In [ ]:
ALIGNED_DIR = OUTPUT_DIR / "_aligned_condition"
ALIGNED_DIR.mkdir(exist_ok=True)

# Containers: year → 2-D float32 array
cond_arrays    = {}   # SEEA-EA score 1–5, NaN = not assessed
invalid_counts = {}   # pixels whose value was not a valid class code (set to NaN)
raw_grid_info  = {}   # grid of the *input* files, used by the quality checks

mapped_mask = ~excl_mask
n_mapped_px = int(mapped_mask.sum())

print(f"Loading condition rasters (MAES reference: {maes_res:.1f} m, {maes_crs})\n")

for year, cond_path in CONDITION_PERIODS.items():
    if not cond_path.exists():
        print(f"  ⚠️  Year {year}: NOT FOUND → {cond_path}")
        continue

    with rasterio.open(cond_path) as src:
        res_m, crs, shape, tf = abs(src.transform.a), str(src.crs), (src.height, src.width), src.transform
    raw_grid_info[year] = {"res": res_m, "crs": crs, "shape": shape}

    same_grid = (
        crs == str(maes_crs)
        and shape == maes_arr.shape
        and np.allclose(tuple(tf)[:6], tuple(maes_tf)[:6], atol=1e-6)
    )

    if ALIGN_CONDITION and not same_grid:
        load_path = ALIGNED_DIR / f"condition_{year}_aligned.tif"
        align_to_maes(cond_path, MAES_RASTER, load_path, RESAMPLE_METHOD)
        grid_tag = f"re-gridded ({RESAMPLE_METHOD.name}) → MAES grid"
    else:
        load_path = cond_path
        grid_tag = "already on MAES grid" if same_grid else "NOT re-gridded (ALIGN_CONDITION = False)"

    with rasterio.open(load_path) as src:
        arr = src.read(1).astype(np.float32)
        nd  = src.nodata if src.nodata is not None else CONDITION_NODATA
    arr = np.where(np.isclose(arr, nd) | np.isnan(arr), np.nan, arr).astype(np.float32)

    if arr.shape != maes_arr.shape:
        raise ValueError(
            f"Condition raster {year} has shape {arr.shape}, MAES has {maes_arr.shape}. "
            "Set ALIGN_CONDITION = True or provide co-registered rasters."
        )

    arr, n_inv = sanitize_condition(arr)
    cond_arrays[year]    = arr
    invalid_counts[year] = n_inv

    assessed = ~np.isnan(arr) & mapped_mask
    n_ass    = int(assessed.sum())
    print(f"  ✅ Year {year}  |  {grid_tag}")
    print(f"     Assessed pixels : {n_ass:,} of {n_mapped_px:,} mapped "
          f"({n_ass / n_mapped_px * 100:.1f} %)")
    if n_inv:
        print(f"     ⚠️  {n_inv:,} pixel(s) outside the valid class codes 1–5 were set to NaN")
    for cc, cl in COND_LABELS.items():
        n   = int(np.sum((arr == cc) & mapped_mask))
        pct = n / n_ass * 100 if n_ass else 0
        print(f"     {cc} – {cl:<22}: {n:>9,} px  ({pct:.1f}%)")
    print()

assert cond_arrays, "No condition raster could be loaded — check CONDITION_PERIODS."
print(f"{len(cond_arrays)}/{len(CONDITION_PERIODS)} condition rasters loaded.")

In [ ]:
# ── 2a. SEEA-EA condition maps for all periods ────────────────────────────────
n_yr = len(cond_arrays)
fig, axes = plt.subplots(1, n_yr, figsize=(7 * n_yr, 5.8), squeeze=False)

for ax, (year, arr) in zip(axes[0], cond_arrays.items()):
    rgb_cond, patches_cond = make_cond_rgb(arr)
    ax.imshow(rgb_cond, interpolation="nearest")
    ax.legend(handles=patches_cond, loc="lower right", fontsize=7,
              title="Condition score", framealpha=0.85)
    ax.set_title(f"SEEA-EA ecosystem condition — {year}", fontsize=11, fontweight="bold")
    ax.axis("off")

plt.suptitle(
    f"Ecosystem Condition (natural ecosystems) — {LIVING_SPACE}",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
save_fig(fig, OUTPUT_DIR / "fig2_condition_maps_all_years.png")
plt.show()

---
## Step 3 — Spatial Overlay and Attribution

**D3.5 §4.2.1.3**

For each MAES ecosystem type $i$, the area-weighted mean condition score is computed
on the assessed pixels:

$$\overline{S}_i = \frac{\sum_{p \in i} S_p \times A_p}{\sum_{p \in i} A_p}$$

together with median, standard deviation, the area and share of each SEEA-EA class,
and the share of the ecosystem type that was assessed.
Ecosystem types that are not scored upstream (e.g. cropland, urban) have no assessed pixels and are reported as **not assessed**.

In [ ]:
# Compute zonal statistics for every year
zonal_by_year = {}   # year → DataFrame

for year, cond_arr in cond_arrays.items():
    df = zonal_condition_stats(
        maes_arr     = maes_arr,
        cond_arr     = cond_arr,
        pixel_ha     = pix_ha,
        unique_codes = unique_codes,
        excl_mask    = excl_mask,
    )
    df["ecosystem_type"] = df["maes_code"].map(code_to_name).fillna("Unknown")
    zonal_by_year[year]  = df
    n_na = int(df["mean_score"].isna().sum())
    print(f"  ✅ Zonal statistics computed for year {year}  "
          f"({len(df) - n_na} assessed, {n_na} not assessed ecosystem type(s))")

print("\nZonal overlay complete.")

---
## Step 4 — Ecosystem Health Indicators

**D3.5 §4.2.1.4**

Three complementary health indicators are derived from the zonal condition statistics:

| Indicator | Description | Equation |
|---|---|---|
| **A** | Mean Condition Score (1–5, higher = better) | $\overline{S}_i$ per ecosystem type |
| **B** | % Assessed Area in Poor / Very Poor condition | $\text{Area}_{\text{class 1–2},i} / \text{Area}_{\text{assessed},i} \times 100$ |
| **C** | Ecosystem Health Index (0–1) | $(\overline{S}_i - S_{\min}) / (S_{\max} - S_{\min})$, with $S_{\min}=1$, $S_{\max}=5$ |

The health status label of an ecosystem type is the SEEA-EA class nearest to its mean score (`health_status`); `health_status_d35` gives the three-level wording of D3.5 Table 5 (Good ≥ 3.5, Moderate 2.5–3.5, Degraded < 2.5).

In [ ]:
# ── Build health indicator table for the primary accounting year ───────────────
if ACCOUNTING_YEAR not in zonal_by_year:
    ACCOUNTING_YEAR = max(zonal_by_year.keys())
    print(f"⚠️  Accounting year adjusted to {ACCOUNTING_YEAR} (nearest available).")

zdf = zonal_by_year[ACCOUNTING_YEAR].copy()

# Indicator A — Mean Condition Score (= 'mean_score'); add health_status label (Table 5)
zdf["health_status"] = zdf["mean_score"].apply(health_label)
zdf["health_status_d35"] = zdf["mean_score"].apply(health_label_d35)   # D3.5 Table 5 wording

# Indicator B — % assessed area in Poor + Very Poor condition ('pct_poor_degraded')
#               already computed in zonal_condition_stats

# Indicator C — Normalised Ecosystem Health Index (0–1)
zdf["health_index"] = compute_health_index(zdf["mean_score"])

# Ecosystem types that could be assessed (used by the figures)
zdf_a = zdf[zdf["mean_score"].notna()].reset_index(drop=True)
not_assessed_types = zdf.loc[zdf["mean_score"].isna(), "ecosystem_type"].tolist()
status_colour = {lab: COND_COLOURS[cc] for cc, lab in COND_LABELS.items()}

# ── Table 5: Mean condition per ecosystem type (D3.5 Table 5) ─────────────────
table5_cols = ["maes_code", "ecosystem_type", "area_ha", "area_assessed_ha", "pct_assessed",
               "mean_score", "median_score", "std_score",
               "health_status", "health_status_d35", "pct_poor_degraded", "health_index"]
table5 = zdf[table5_cols].copy()

print(f"📋 Table 5 — Mean condition per ecosystem type (SEEA-EA score, {ACCOUNTING_YEAR}):")
display(table5)
if not_assessed_types:
    print("ℹ️  Not assessed (no condition score in the input raster): "
          + ", ".join(not_assessed_types))

out_t5 = OUTPUT_DIR / f"table5_mean_condition_{ACCOUNTING_YEAR}.csv"
table5.to_csv(out_t5, index=False)
print(f"Saved → {out_t5.name}")

In [ ]:
# ── Full condition account (including area breakdown by class) ─────────────────
cond_account_cols = [
    "maes_code", "ecosystem_type", "area_ha", "area_assessed_ha", "pct_assessed",
    "area_ha_very_good", "area_ha_good", "area_ha_fair", "area_ha_poor", "area_ha_very_poor",
    "pct_very_good", "pct_good", "pct_fair", "pct_poor", "pct_very_poor",
    "pct_poor_degraded",
    "mean_score", "median_score", "std_score", "min_score", "max_score",
    "health_status", "health_status_d35", "health_index",
]
cond_account = zdf[cond_account_cols].copy()
print("📋 Full SEEA-EA Condition Account:")
display(cond_account)

out_cond = OUTPUT_DIR / f"table_condition_account_{ACCOUNTING_YEAR}.csv"
cond_account.to_csv(out_cond, index=False)
print(f"Saved → {out_cond.name}")

In [ ]:
# ── Figure: Indicator A — Mean condition score per ecosystem type ─────────────
if zdf_a.empty:
    print("No assessed ecosystem type — Indicator A figure skipped.")
else:
    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.9 * len(zdf_a) + 1.8)))
    bar_colours = [status_colour[s] for s in zdf_a["health_status"]]
    bars = ax.barh(zdf_a["ecosystem_type"], zdf_a["mean_score"],
                   color=bar_colours, edgecolor="grey", linewidth=0.5)
    # Error bars (±1 std)
    ax.errorbar(zdf_a["mean_score"], zdf_a["ecosystem_type"],
                xerr=zdf_a["std_score"], fmt="none",
                ecolor="black", elinewidth=1, capsize=4)
    # Condition class bands (class c spans c − 0.5 … c + 0.5)
    for cc, col in COND_COLOURS.items():
        ax.axvspan(cc - 0.5, cc + 0.5, alpha=0.07, color=col)
    for x in (1.5, 2.5, 3.5, 4.5):
        ax.axvline(x, color="grey", lw=0.8, ls="--")
    # Annotate mean
    for bar, (_, row) in zip(bars, zdf_a.iterrows()):
        ax.text(min(row["mean_score"] + row["std_score"] + 0.05, 5.3),
                bar.get_y() + bar.get_height() / 2,
                f"{row['mean_score']:.2f}", va="center", fontsize=8)
    ax.set_xlim(0.5, 5.5)
    ax.set_xticks(range(COND_MIN, COND_MAX + 1))
    ax.set_xticklabels([f"{c}\n{COND_LABELS[c].split(' /')[0]}" for c in range(COND_MIN, COND_MAX + 1)],
                       fontsize=8)
    ax.set_xlabel("Mean SEEA-EA condition score  (higher = better condition)", fontsize=10)
    ax.set_title(
        f"Indicator A — Mean Condition Score per Ecosystem Type\n"
        f"{LIVING_SPACE}  |  {ACCOUNTING_YEAR}",
        fontsize=11, fontweight="bold"
    )
    plt.tight_layout()
    if not_assessed_types:
        fig.text(0.01, -0.03, "Not assessed: " + ", ".join(not_assessed_types),
                 fontsize=8, style="italic")
    save_fig(fig, OUTPUT_DIR / f"fig3_indicator_A_mean_score_{ACCOUNTING_YEAR}.png")
    plt.show()

In [ ]:
# ── Figure: Indicator B — % area in each condition class (stacked bar) ────────
if zdf_a.empty:
    print("No assessed ecosystem type — Indicator B figure skipped.")
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    bottom = np.zeros(len(zdf_a))
    for cc, cl in COND_LABELS.items():            # 5 (best) at the bottom … 1 (worst) on top
        vals = zdf_a[f"pct_{COND_KEYS[cc]}"].fillna(0).values
        bars = ax.bar(zdf_a["ecosystem_type"], vals,
                      bottom=bottom, label=f"{cc} – {cl}",
                      color=COND_COLOURS[cc], edgecolor="grey", linewidth=0.4)
        txt_col = "black" if cc in (4, 3) else "white"
        for bar, v, bot in zip(bars, vals, bottom):
            if v > 8:   # annotate slices > 8 %
                ax.text(bar.get_x() + bar.get_width() / 2, bot + v / 2,
                        f"{v:.0f}%", ha="center", va="center",
                        fontsize=8, color=txt_col, fontweight="bold")
        bottom += vals
    # Indicator B value (Poor + Very Poor) above each bar
    for x, v in enumerate(zdf_a["pct_poor_degraded"]):
        ax.text(x, 102, f"B = {v:.0f}%", ha="center", fontsize=8, fontweight="bold")
    ax.set_ylabel("% of assessed ecosystem area", fontsize=10)
    ax.set_ylim(0, 112)
    ax.yaxis.set_major_formatter(PercentFormatter())
    ax.set_title(
        f"Indicator B — Area Share by Condition Class per Ecosystem Type\n"
        f"{LIVING_SPACE}  |  {ACCOUNTING_YEAR}   (B = Poor + Very Poor)",
        fontsize=11, fontweight="bold"
    )
    ax.legend(title="Condition class", fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
    plt.xticks(rotation=30, ha="right", fontsize=9)
    plt.tight_layout()
    save_fig(fig, OUTPUT_DIR / f"fig4_indicator_B_pct_poor_degraded_{ACCOUNTING_YEAR}.png")
    plt.show()

In [ ]:
# ── Figure: Indicator C — Ecosystem Health Index (0–1) ────────────────────────
if zdf_a.empty:
    print("No assessed ecosystem type — Indicator C figure skipped.")
else:
    health_colours = [status_colour[s] for s in zdf_a["health_status"]]

    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.8 * len(zdf_a) + 1.8)))
    bars = ax.barh(zdf_a["ecosystem_type"], zdf_a["health_index"],
                   color=health_colours, edgecolor="grey", linewidth=0.5)
    # SEEA-EA class bands expressed on the 0–1 index (class boundaries at score x.5)
    for cc, col in COND_COLOURS.items():
        lo = max(0.0, (cc - 1.5 - COND_MIN + 1) / (COND_MAX - COND_MIN))
        hi = min(1.0, (cc - 0.5 - COND_MIN + 1) / (COND_MAX - COND_MIN))
        ax.axvspan(lo, hi, alpha=0.07, color=col)
        ax.text((lo + hi) / 2, 1.01, COND_LABELS[cc].split(" /")[0], ha="center",
                va="bottom", fontsize=7, color="dimgrey",
                transform=ax.get_xaxis_transform())
    for b in (0.125, 0.375, 0.625, 0.875):
        ax.axvline(b, color="grey", lw=0.8, ls="--")
    for bar, v in zip(bars, zdf_a["health_index"]):
        ax.text(min(v + 0.015, 0.97),
                bar.get_y() + bar.get_height() / 2,
                f"{v:.3f}", va="center", fontsize=9)
    ax.set_xlabel("Health Index  (0 = worst, 1 = best)", fontsize=10)
    ax.set_xlim(0, 1.05)
    ax.set_title(
        f"Indicator C — Ecosystem Health Index (normalised SEEA-EA score)\n"
        f"{LIVING_SPACE}  |  {ACCOUNTING_YEAR}",
        fontsize=11, fontweight="bold", pad=22
    )
    plt.tight_layout()
    if not_assessed_types:
        fig.text(0.01, -0.03, "Not assessed: " + ", ".join(not_assessed_types),
                 fontsize=8, style="italic")
    save_fig(fig, OUTPUT_DIR / f"fig5_indicator_C_health_index_{ACCOUNTING_YEAR}.png")
    plt.show()

In [ ]:
# ── Spatial Ecosystem Health Index: maps + GeoTIFF export ─────────────────────
cond_ref = cond_arrays[ACCOUNTING_YEAR]

# (1) Pixel-level health index — directly from the 1–5 condition score
health_pixel = compute_health_index(cond_ref).astype(np.float32)
health_pixel[excl_mask] = np.nan

# (2) Ecosystem-type-level health index — every pixel gets its ecosystem type's value
health_type = np.full(maes_arr.shape, np.nan, dtype=np.float32)
for code, hi in zip(zdf["maes_code"], zdf["health_index"]):
    if pd.notna(hi):
        health_type[(maes_arr == code) & (~excl_mask)] = hi

cmap_health = matplotlib.colormaps["RdYlGn"].copy()
cmap_health.set_bad(NOT_ASSESSED_RGB)

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

rgb_cond, patches_cond = make_cond_rgb(cond_ref)
axes[0].imshow(rgb_cond, interpolation="nearest")
axes[0].legend(handles=patches_cond, loc="lower right", fontsize=7,
               title="Condition score", framealpha=0.85)
axes[0].set_title(f"SEEA-EA condition (pixel-level) — {ACCOUNTING_YEAR}",
                  fontsize=11, fontweight="bold")
axes[0].axis("off")

im1 = axes[1].imshow(np.ma.masked_invalid(health_pixel), cmap=cmap_health,
                     vmin=0, vmax=1, interpolation="nearest")
plt.colorbar(im1, ax=axes[1], fraction=0.04, pad=0.04, label="Health Index (0–1)")
axes[1].set_title(f"Ecosystem Health Index (pixel-level) — {ACCOUNTING_YEAR}",
                  fontsize=11, fontweight="bold")
axes[1].axis("off")

im2 = axes[2].imshow(np.ma.masked_invalid(health_type), cmap=cmap_health,
                     vmin=0, vmax=1, interpolation="nearest")
plt.colorbar(im2, ax=axes[2], fraction=0.04, pad=0.04, label="Health Index (0–1)")
axes[2].set_title(f"Ecosystem Health Index (by ecosystem type) — {ACCOUNTING_YEAR}",
                  fontsize=11, fontweight="bold")
axes[2].axis("off")

plt.suptitle(f"Ecosystem Condition & Health Index — {LIVING_SPACE}  (grey = not assessed)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig(fig, OUTPUT_DIR / f"fig6_health_index_map_{ACCOUNTING_YEAR}.png")
plt.show()


# ── Export the Ecosystem Health product as GeoTIFFs (MAES grid) ───────────────
def write_health_tif(arr: np.ndarray, path: Path, description: str) -> None:
    with rasterio.open(MAES_RASTER) as ref:
        profile = ref.profile.copy()
    profile.update(dtype="float32", count=1, nodata=-9999.0, compress="lzw")
    out = np.where(np.isnan(arr), -9999.0, arr).astype(np.float32)
    with rasterio.open(path, "w", **profile) as dst:
        dst.write(out, 1)
        dst.update_tags(1, description=description,
                        values="0 = worst, 1 = best, -9999 = not assessed",
                        formula="(SEEA-EA condition score - 1) / 4",
                        framework="UN SEEA EA 2021",
                        source="LandShift D3.5 section 4.2")
    print(f"   GeoTIFF saved → {path.name}")

write_health_tif(health_pixel,
                 OUTPUT_DIR / f"ecosystem_health_index_pixel_{ACCOUNTING_YEAR}.tif",
                 "Ecosystem Health Index, pixel level")
write_health_tif(health_type,
                 OUTPUT_DIR / f"ecosystem_health_index_by_type_{ACCOUNTING_YEAR}.tif",
                 "Ecosystem Health Index, mean per MAES ecosystem type")

---
## Step 5 — Integration into SEEA-EA Condition Accounts

**D3.5 §4.2.1.5**

When multiple accounting periods are available, the change in mean condition score
between the first and the last period is recorded per ecosystem type
(Table 6 of D3.5), together with the change in the Health Index and in the share of
Poor / Very Poor area.  Because the SEEA-EA scale is oriented *higher = better*,
a **positive change means improving** condition (the reverse of the former ESAI-based account).
At pixel level, the change in condition class is summarised for pixels assessed in both periods.

> ⚠️ Trends are only meaningful when all periods were produced with the same indicator weights,
> class thresholds and 2000–2015 reference baseline.

In [ ]:
years_sorted = sorted(zonal_by_year.keys())
change_summary = None   # filled in by the pixel-level change cell below

if len(years_sorted) >= 2:
    year_first = years_sorted[0]
    year_last  = years_sorted[-1]

    df_first = zonal_by_year[year_first][["maes_code", "ecosystem_type",
                                          "mean_score", "pct_poor_degraded"]].copy()
    df_last  = zonal_by_year[year_last][["maes_code", "mean_score",
                                         "pct_poor_degraded"]].copy()

    # Re-derive health index for both periods
    df_first["health_index"] = compute_health_index(df_first["mean_score"])
    df_last["health_index"]  = compute_health_index(df_last["mean_score"])

    trend = df_first.merge(
        df_last, on="maes_code", suffixes=(f"_{year_first}", f"_{year_last}")
    )
    trend["delta_score"]             = (trend[f"mean_score_{year_last}"] -
                                        trend[f"mean_score_{year_first}"]).round(4)
    trend["delta_health_index"]      = (trend[f"health_index_{year_last}"] -
                                        trend[f"health_index_{year_first}"]).round(4)
    trend["delta_pct_poor_degraded"] = (trend[f"pct_poor_degraded_{year_last}"] -
                                        trend[f"pct_poor_degraded_{year_first}"]).round(2)

    def trend_label(delta_score):
        """Higher SEEA-EA score = better condition → positive change = improving."""
        if pd.isna(delta_score):
            return "Not assessed"
        if abs(delta_score) < TREND_STABLE_TOL:
            return "Stable"
        return "Improving" if delta_score > 0 else "Degrading"

    trend["trend"] = trend["delta_score"].apply(trend_label)

    # ── Table 6 (D3.5) ───────────────────────────────────────────────────────
    table6_cols = [
        "maes_code", "ecosystem_type",
        f"mean_score_{year_first}", f"mean_score_{year_last}", "delta_score",
        f"health_index_{year_first}", f"health_index_{year_last}", "delta_health_index",
        "delta_pct_poor_degraded", "trend",
    ]
    table6 = trend[table6_cols].copy()

    print(f"📋 Table 6 — Change in Mean Condition Score: {year_first} → {year_last}  (D3.5 Table 6):")
    display(table6)

    out_t6 = OUTPUT_DIR / f"table6_condition_trend_{year_first}_{year_last}.csv"
    table6.to_csv(out_t6, index=False)
    print(f"Saved → {out_t6.name}")

else:
    print("ℹ️  Only one accounting period provided — trend table skipped.")
    trend = None

In [ ]:
TREND_COLOURS = {"Improving": "#2ca25f", "Stable": "#cccccc", "Degrading": "#d73027"}

if trend is not None and trend["delta_score"].notna().any():
    trend_plot = trend.dropna(subset=["delta_score"]).reset_index(drop=True)
    t_colours  = [TREND_COLOURS[t] for t in trend_plot["trend"]]

    # ── Figure: condition trend per ecosystem type ────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, max(3.8, 0.8 * len(trend_plot) + 2.2)))

    for ax, col, ttl, xl in [
        (axes[0], "delta_score",        "Change in Mean Condition Score\n(positive = improving)",
         f"Δ score  ({year_first} → {year_last})"),
        (axes[1], "delta_health_index", "Change in Health Index\n(positive = improving)",
         f"Δ Health Index  ({year_first} → {year_last})"),
    ]:
        bars = ax.barh(trend_plot["ecosystem_type"], trend_plot[col],
                       color=t_colours, edgecolor="grey", linewidth=0.5)
        ax.axvline(0, color="black", lw=0.8)
        span = max(trend_plot[col].abs().max(), 1e-3)
        for bar, v in zip(bars, trend_plot[col]):
            ax.text(v + (0.02 * span if v >= 0 else -0.02 * span),
                    bar.get_y() + bar.get_height() / 2,
                    f"{v:+.3f}", va="center", ha="left" if v >= 0 else "right", fontsize=8)
        ax.set_xlim(-span * 1.4, span * 1.4)
        ax.set_xlabel(xl, fontsize=10)
        ax.set_title(ttl, fontsize=11, fontweight="bold")
        ax.yaxis.set_tick_params(labelsize=9)

    legend_patches = [mpatches.Patch(color=c, label=l) for l, c in TREND_COLOURS.items()]
    fig.legend(handles=legend_patches, loc="lower center",
               ncol=3, fontsize=9, framealpha=0.85,
               title="Trend", bbox_to_anchor=(0.5, -0.08))

    plt.suptitle(
        f"Ecosystem Health Trends — {LIVING_SPACE}  |  {year_first} → {year_last}",
        fontsize=12, fontweight="bold"
    )
    plt.tight_layout()
    save_fig(fig, OUTPUT_DIR / f"fig7_condition_trend_{year_first}_{year_last}.png")
    plt.show()

    # ── Figure: multi-year mean-score time-series per ecosystem type ──────────
    ts = pd.DataFrame({yr: zonal_by_year[yr].set_index("maes_code")["mean_score"]
                       for yr in years_sorted}).dropna(how="all")
    palette_ts = get_cmap_n("tab10", max(len(ts), 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    for j, (code, row) in enumerate(ts.iterrows()):
        ax.plot(years_sorted, row.values, marker="o", lw=2, color=palette_ts(j),
                label=code_to_name.get(code, str(code)))
    for cc, col in COND_COLOURS.items():
        ax.axhspan(cc - 0.5, cc + 0.5, alpha=0.07, color=col)
    ax.set_xlabel("Year", fontsize=10)
    ax.set_ylabel("Mean SEEA-EA condition score (higher = better)", fontsize=10)
    ax.set_title(f"Mean Condition Score Trend by Ecosystem Type — {LIVING_SPACE}",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
    ax.set_xticks(years_sorted)
    ax.set_ylim(COND_MIN - 0.5, COND_MAX + 0.5)
    plt.tight_layout()
    save_fig(fig, OUTPUT_DIR / "fig8_condition_timeseries.png")
    plt.show()
else:
    print("ℹ️  No comparable assessed ecosystem type across periods — trend figures skipped.")

In [ ]:
# ── Spatial change map (change in condition class between first and last period) ──
if len(years_sorted) >= 2:
    delta_map = cond_arrays[year_last] - cond_arrays[year_first]   # NaN where either is not assessed
    delta_map[excl_mask] = np.nan
    both = ~np.isnan(delta_map)
    n_both = int(both.sum())

    if n_both:
        change_summary = {
            "n_px"        : n_both,
            "pct_improved": float(np.mean(delta_map[both] > 0) * 100),
            "pct_stable"  : float(np.mean(delta_map[both] == 0) * 100),
            "pct_degraded": float(np.mean(delta_map[both] < 0) * 100),
        }
        print(f"Pixels assessed in both periods: {n_both:,}")
        print(f"   Improved : {change_summary['pct_improved']:5.1f} %")
        print(f"   Stable   : {change_summary['pct_stable']:5.1f} %")
        print(f"   Degraded : {change_summary['pct_degraded']:5.1f} %")

        vmax_delta = max(1, int(np.ceil(np.nanpercentile(np.abs(delta_map), 99))))
        cmap_delta = matplotlib.colormaps["RdYlGn"].copy()   # green = improving
        cmap_delta.set_bad(NOT_ASSESSED_RGB)

        fig, ax = plt.subplots(figsize=(8, 6))
        im = ax.imshow(np.ma.masked_invalid(delta_map), cmap=cmap_delta,
                       vmin=-vmax_delta, vmax=vmax_delta, interpolation="nearest")
        cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
        cbar.set_label(f"Δ condition class  ({year_first} → {year_last})\n"
                       "positive = improving, negative = degrading", fontsize=8)
        ax.set_title(
            f"Pixel-level change in condition class  ({year_first} → {year_last})\n"
            f"{LIVING_SPACE}  (grey = not assessed in both periods)",
            fontsize=11, fontweight="bold"
        )
        ax.axis("off")
        plt.tight_layout()
        save_fig(fig, OUTPUT_DIR / f"fig9_delta_condition_{year_first}_{year_last}.png")
        plt.show()
    else:
        print("ℹ️  No pixel assessed in both periods — change map skipped.")
else:
    print("ℹ️  Only one period — change map skipped.")

---
## Section 6 — Quality Checks

**D3.5 §4.2.3** — *Technical Considerations*

Five checks implement the requirements listed in §4.2.3:

1. Spatial grid consistency (MAES vs. condition rasters)
2. Condition value validity (class codes 1–5 only)
3. Assessment coverage of the condition account (not-assessed / partially assessed ecosystem types)
4. Temporal comparability of input datasets
5. Condition class coverage (all five classes represented)

In [ ]:
print("═" * 65)
print("TECHNICAL QUALITY CHECKS  (D3.5 §4.2.3)")
print("═" * 65)
issues = []

# 1. Spatial grid consistency
print(f"\n1. Spatial grid consistency  (MAES reference: {maes_res:.1f} m, {maes_crs})")
for yr, info in raw_grid_info.items():
    ok = (abs(info["res"] - maes_res) < 0.5
          and info["crs"] == str(maes_crs)
          and info["shape"] == maes_arr.shape)
    if ok:
        tag = "✅"
    elif ALIGN_CONDITION:
        tag = "↻ input differs from MAES grid → re-gridded (nearest neighbour)"
    else:
        tag = "⚠️  MISMATCH"
        issues.append(f"Grid mismatch for condition {yr}: {info['res']:.1f} m / {info['shape']} "
                      f"vs {maes_res:.1f} m / {maes_arr.shape}")
    print(f"   Condition {yr}: {info['res']:6.1f} m  {info['shape']}  {tag}")

# 2. Condition value validity
print("\n2. Condition value validity  (expected class codes: 1, 2, 3, 4, 5)")
for yr, arr in cond_arrays.items():
    n_inv = invalid_counts.get(yr, 0)
    found = sorted(int(v) for v in np.unique(arr[~np.isnan(arr)]))
    tag = "✅" if n_inv == 0 else f"⚠️  {n_inv:,} invalid pixel(s) set to NaN"
    if n_inv:
        issues.append(f"Condition {yr}: {n_inv:,} pixel(s) outside class codes 1–5 "
                      "(wrong raster, or resampled with a non-nearest method?).")
    print(f"   Condition {yr}: classes found = {found}  {tag}")

# 3. Assessment coverage of the condition account
print(f"\n3. Assessment coverage ({ACCOUNTING_YEAR})")
na_rows = cond_account[cond_account["mean_score"].isna()]
if not na_rows.empty:
    print("   ℹ️  Not assessed (no scored pixel; expected for non-natural ecosystems "
          "such as cropland, urban, water):")
    for _, r in na_rows.iterrows():
        print(f"        - {r['ecosystem_type']}  ({r['area_ha']:,.1f} ha)")
partial = cond_account[cond_account["mean_score"].notna()
                       & (cond_account["pct_assessed"] < MIN_ASSESSED_PCT)]
if not partial.empty:
    display(partial[["ecosystem_type", "area_ha", "pct_assessed"]])
    issues.append(f"{len(partial)} assessed ecosystem type(s) have < {MIN_ASSESSED_PCT:.0f} % "
                  "of their area scored.")
else:
    print(f"   ✅ All assessed ecosystem types have ≥ {MIN_ASSESSED_PCT:.0f} % of their area scored.")
if cond_account["mean_score"].notna().sum() == 0:
    issues.append("No ecosystem type could be assessed.")

# 4. Temporal comparability
print("\n4. Temporal comparability")
if len(years_sorted) >= 2:
    shapes   = {yr: cond_arrays[yr].shape for yr in years_sorted}
    all_same = len(set(shapes.values())) == 1
    tag = "✅" if all_same else "⚠️  SHAPE MISMATCH"
    print(f"   Array shapes across periods: {shapes}  {tag}")
    if not all_same:
        issues.append("Condition arrays have inconsistent shapes across periods.")
    assessed_sets = {yr: set(zonal_by_year[yr].loc[zonal_by_year[yr]["mean_score"].notna(), "maes_code"])
                     for yr in years_sorted}
    same_sets = len({frozenset(s) for s in assessed_sets.values()}) == 1
    print(f"   Assessed ecosystem types identical across periods: {'✅' if same_sets else '⚠️  DIFFERENT'}")
    if not same_sets:
        issues.append("The set of assessed ecosystem types differs between periods.")
    print("   ℹ️  Reminder: all periods must share the 2000–2015 reference baseline, indicator "
          "weights and class thresholds (cannot be verified from the rasters).")
else:
    print("   ℹ️  Only one period — temporal check skipped.")

# 5. Condition class coverage
print(f"\n5. Condition class coverage in primary year ({ACCOUNTING_YEAR})")
cond_chk = cond_arrays.get(ACCOUNTING_YEAR)
if cond_chk is not None:
    for cc, cl in COND_LABELS.items():
        n = int(np.sum((cond_chk == cc) & mapped_mask))
        tag = "✅" if n > 0 else "⚠️  CLASS ABSENT"
        print(f"   {cc} – {cl:<22}: {n:>9,} px  {tag}")
        if n == 0:
            issues.append(f"Condition class {cc} ('{cl}') has zero pixels in {ACCOUNTING_YEAR}.")

# Summary
print("\n" + "═" * 65)
if issues:
    print(f"⚠️  {len(issues)} issue(s) require attention:")
    for i, iss in enumerate(issues, 1):
        print(f"   {i}. {iss}")
else:
    print("✅ All checks passed — condition account is ready for SEEA-EA reporting.")
print("═" * 65)

---
## Section 7 — Grand-Total Summary

In [ ]:
def fmt(x, width=10, prec=3):
    return f"{x:>{width}.{prec}f}" if pd.notna(x) else f"{'n/a':>{width}}"

print("GRAND SUMMARY — SEEA-EA Ecosystem Condition Account")
print("-" * 65)
print(f"Living Space      : {LIVING_SPACE}")
print(f"Accounting year   : {ACCOUNTING_YEAR}")
print(f"Condition periods : {years_sorted}")
print(f"Total mapped area : {total_area:,.2f} ha  ({total_area/100:,.2f} km²)")
print(f"Ecosystem types   : {len(extent_table)}  "
      f"({int(zdf['mean_score'].notna().sum())} assessed, {len(not_assessed_types)} not assessed)")
print()

# Condition class totals for the accounting year (assessed pixels inside the mapped extent)
cond_yr = cond_arrays.get(ACCOUNTING_YEAR)
if cond_yr is not None:
    n_assessed_px = int(np.sum(~np.isnan(cond_yr) & mapped_mask))
    print(f"  {'Condition class':<28} {'Area (ha)':>12}   {'Share':>7}")
    print("  " + "-" * 51)
    for cc, cl in COND_LABELS.items():
        n_px = int(np.sum((cond_yr == cc) & mapped_mask))
        area = n_px * pix_ha
        pct  = n_px / n_assessed_px * 100 if n_assessed_px else 0
        print(f"  {cc} – {cl:<24} {area:>12,.2f}   {pct:>6.1f}%")
    print(f"  {'Assessed area':<28} {n_assessed_px * pix_ha:>12,.2f}   "
          f"({n_assessed_px / n_mapped_px * 100:.1f} % of mapped area)")
    print()

print(f"  {'Ecosystem Type':<32} {'Mean score':>10}  {'Health Index':>13}  {'% Poor/VP':>10}  {'Status':<22}")
print("  " + "-" * 94)
for _, row in table5.iterrows():
    print(f"  {row['ecosystem_type']:<32} {fmt(row['mean_score'], 10, 3)}  "
          f"{fmt(row['health_index'], 13, 3)}  {fmt(row['pct_poor_degraded'], 10, 1)}  "
          f"{row['health_status']:<22}")

print()
print(f"All outputs in: {OUTPUT_DIR.resolve()}")

---
## Section 8 — Processing Report

In [ ]:
ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def fnum(x, nd=4, signed=False):
    if pd.isna(x):
        return "n/a"
    return f"{x:+.{nd}f}" if signed else f"{x:.{nd}f}"

lines = [
    "# LandShift D3.5 — Section 4.2 Ecosystem Health Processing Report",
    "",
    "## Run metadata",
    "",
    "| Key | Value |",
    "|---|---|",
    f"| Generated | {ts} |",
    f"| Living Space | {LIVING_SPACE} |",
    f"| Accounting year | {ACCOUNTING_YEAR} |",
    f"| Condition periods | {years_sorted} |",
    f"| Demo mode | {DEMO_MODE} |",
    "",
    "## Inputs",
    "",
    "| Role | Path |",
    "|---|---|",
    f"| MAES raster | `{MAES_RASTER}` |",
    f"| MAES lookup | `{MAES_LOOKUP_CSV}` |",
]
for yr, p in CONDITION_PERIODS.items():
    lines.append(f"| SEEA-EA condition {yr} | `{p}` |")

lines += [
    "",
    "## Spatial parameters",
    "",
    f"- CRS: `{maes_crs}`",
    f"- Resolution: {maes_res:.1f} m",
    f"- Pixel area: {pix_ha:.6f} ha",
    f"- Condition re-gridding enabled: `{ALIGN_CONDITION}` (resampling: `{RESAMPLE_METHOD.name}`)",
    "",
    "## Condition methodology",
    "",
    "- Source: HE-LandShift WP3 `ecosystem_condition.ipynb` (SEEA-EA 5-point score; reference period 2000–2015)",
    "- Indicators (upstream): NDVI deviation, canopy-cover deficit, fragmentation, human pressure",
    "- Scale: " + ", ".join(f"{cc} = {cl}" for cc, cl in COND_LABELS.items()),
    "- Only natural ecosystems are scored; other ecosystem types are reported as *not assessed*",
    f"- Health index: (score − {COND_MIN}) / ({COND_MAX} − {COND_MIN}); indicator B = classes {list(DEGRADED_CLASSES)}",
    "",
    "## Ecosystem Extent Account (Table 4)",
    "",
    f"Total mapped area: **{total_area:,.2f} ha**  ({total_area/100:,.2f} km²)  ",
    f"Ecosystem types: **{len(extent_table)}**",
    "",
    "| Ecosystem Type | Area (ha) | Share (%) |",
    "|---|---|---|",
]
for _, row in extent_table.iterrows():
    lines.append(f"| {row['ecosystem_type']} | {row['area_ha']:,.2f} | {row['area_pct']:.1f} |")

lines += [
    "",
    f"## Mean Condition Account (Table 5, {ACCOUNTING_YEAR})",
    "",
    "| Ecosystem Type | Assessed (%) | Mean score | Health Status | D3.5 Status | % Poor/Very Poor | Health Index |",
    "|---|---|---|---|---|---|---|",
]
for _, row in table5.iterrows():
    lines.append(
        f"| {row['ecosystem_type']} | {fnum(row['pct_assessed'], 1)} | {fnum(row['mean_score'])} | "
        f"{row['health_status']} | {row['health_status_d35']} | {fnum(row['pct_poor_degraded'], 1)} | {fnum(row['health_index'])} |"
    )

if trend is not None:
    lines += [
        "",
        f"## Condition Trend Account (Table 6, {year_first} → {year_last})",
        "",
        f"| Ecosystem Type | Score {year_first} | Score {year_last} | Δ score | Trend |",
        "|---|---|---|---|---|",
    ]
    for _, row in table6.iterrows():
        lines.append(
            f"| {row['ecosystem_type']} | {fnum(row[f'mean_score_{year_first}'])} | "
            f"{fnum(row[f'mean_score_{year_last}'])} | {fnum(row['delta_score'], 4, True)} | {row['trend']} |"
        )
    if change_summary is not None:
        lines += [
            "",
            f"Pixel-level change in condition class ({change_summary['n_px']:,} pixels assessed in both periods): "
            f"**{change_summary['pct_improved']:.1f} %** improved, "
            f"**{change_summary['pct_stable']:.1f} %** stable, "
            f"**{change_summary['pct_degraded']:.1f} %** degraded.",
        ]

lines += [
    "",
    "## Quality check summary",
    "",
]
lines += [f"- ⚠️ {iss}" for iss in issues] if issues else ["- ✅ All checks passed."]

lines += [
    "",
    "## Output files",
    "",
    f"Directory: `{OUTPUT_DIR.resolve()}/`",
    "",
    "| File | Description |",
    "|---|---|",
    f"| `table4_ecosystem_extent_{ACCOUNTING_YEAR}.csv` | SEEA-EA Ecosystem Extent Account |",
    f"| `table5_mean_condition_{ACCOUNTING_YEAR}.csv` | Mean condition score & Health Indicators (Table 5) |",
    f"| `table_condition_account_{ACCOUNTING_YEAR}.csv` | Full SEEA-EA Condition Account |",
    f"| `table6_condition_trend_*.csv` | Condition Trend Account (Table 6) |",
    f"| `ecosystem_health_index_pixel_{ACCOUNTING_YEAR}.tif` | Ecosystem Health Index, pixel level (GeoTIFF, 0–1) |",
    f"| `ecosystem_health_index_by_type_{ACCOUNTING_YEAR}.tif` | Ecosystem Health Index, per MAES ecosystem type (GeoTIFF, 0–1) |",
    f"| `fig1_*.png` | MAES extent map |",
    f"| `fig2_*.png` | SEEA-EA condition maps (all years) |",
    f"| `fig3_*.png` | Indicator A — Mean condition score per ecosystem type |",
    f"| `fig4_*.png` | Indicator B — % area by condition class |",
    f"| `fig5_*.png` | Indicator C — Ecosystem Health Index |",
    f"| `fig6_*.png` | Spatial Health Index maps (pixel level and by type) |",
    f"| `fig7_*.png` | Condition trend bar charts |",
    f"| `fig8_*.png` | Mean condition score time-series per ecosystem type |",
    f"| `fig9_*.png` | Pixel-level change in condition class |",
    "",
    "---",
    "*LandShift — Horizon Europe Grant Agreement No. 101182007*  ",
    "*D3.5 Ecosystem services & landscape ecology — Section 4.2*  ",
    "*University of Geneva (UNIGE)*",
]

rpt = OUTPUT_DIR / f"processing_report_{ACCOUNTING_YEAR}.md"
rpt.write_text("\n".join(lines), encoding="utf-8")
print(f"Processing report saved → {rpt.name}")
print(f"\n✅  Section 4.2 notebook complete.")
print(f"    All outputs in: {OUTPUT_DIR.resolve()}")

---
## FAIR Compliance Summary

| Principle | Implementation |
|-----------|----------------|
| **Findable** | Year-stamped filenames; Markdown report indexes every output file |
| **Accessible** | Open formats only: GeoTIFF input/output, CSV + PNG + MD output |
| **Interoperable** | MAES (EEA standard) and SEEA-EA table structure (UN standard); condition classes follow the UN SEEA EA 5-point scale as produced by the LandShift WP3 condition workflow |
| **Reusable** | Section 1 isolates all user parameters; helper functions documented with docstrings; `DEMO_MODE` enables testing without real data; `CONDITION_PERIODS` dict supports arbitrary multi-year inputs |

All Tier 2 and Tier 3 notebooks will be published in the **LandShift Zenodo repository**
at M30 (April 2027) with a DOI, as required by Horizon Europe.

---
*LandShift — Horizon Europe Grant Agreement No. 101182007*  
*D3.5 Ecosystem services & landscape ecology — Section 4.2*  
*University of Geneva (UNIGE)*